# E-commerce Web Data Extraction & Dataset Preparation

## Project Overview

This project demonstrates a professional workflow for extracting publicly
available e-commerce product data, transforming the raw API response into a
structured dataset, performing data-quality checks, and exporting the cleaned
dataset for analysis.

### Objectives

- Identify a structured source of publicly available product data.
- Extract product records programmatically.
- Handle API pagination.
- Transform nested JSON fields into tabular columns.
- Clean and standardize the extracted data.
- Validate the quality and completeness of the dataset.
- Export the final dataset in CSV and Excel formats.

### Technologies

- Python
- Requests
- Pandas
- JSON
- Google Colab

## 1. Environment Setup

The extraction workflow uses `requests` to communicate with the public API
and `pandas` to transform and analyze the extracted records.

Additional libraries will be introduced only when they are required for
specific stages of the project.

In [ ]:
# Import libraries required for API requests and data manipulation
import requests
import pandas as pd
import json
import time

## 2. Define the Data Source

WooCommerce provides a public Store API for customer-facing product data.
The product endpoint returns published product information in JSON format
and does not require API authentication.

For this project, we will use the product collection endpoint as the
primary data source.

The endpoint supports pagination, allowing large product collections to be
retrieved in manageable batches.

In [ ]:
# Public product endpoint for the Male Vegadish catalogue
BASE_URL = "https://malevegadish.com/wp-json/wc/store/v1/products"

# Start small while validating the API response
RECORDS_PER_PAGE = 3

In [ ]:
import requests
import time

params = {
    "page": 1,
    "per_page": RECORDS_PER_PAGE
}

try:
    start_time = time.time()

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=(10, 90)
    )

    elapsed = time.time() - start_time

    print(f"HTTP Status Code: {response.status_code}")
    print(f"Response Time: {elapsed:.2f} seconds")
    print(f"Content Type: {response.headers.get('Content-Type')}")

    response.raise_for_status()

    products = response.json()

    print(f"Products Returned: {len(products)}")

except requests.exceptions.Timeout:
    print("The product API did not respond within 90 seconds.")

except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")

except ValueError:
    print("The server responded, but the response was not valid JSON.")

HTTP Status Code: 200
Response Time: 10.12 seconds
Content Type: application/json; charset=UTF-8
Products Returned: 3


In [ ]:
# Display the first product returned by the API
# This lets us inspect the real JSON structure before designing the extractor.

import json

print(json.dumps(products[0], indent=2))

{
  "id": 12192,
  "name": "\u05de\u05d3\u05d1\u05e7\u05ea \u05e1\u05de\u05dc \u05dc\u05e8\u05db\u05d1 THANK YOU HASHEM&quot; ABS&quot;",
  "slug": "%d7%9e%d7%93%d7%91%d7%a7%d7%aa-%d7%a1%d7%9e%d7%9c-%d7%9c%d7%a8%d7%9b%d7%91-thank-you-hashem-abs",
  "parent": 0,
  "type": "simple",
  "variation": "",
  "permalink": "https://malevegadish.com/product/%d7%9e%d7%93%d7%91%d7%a7%d7%aa-%d7%a1%d7%9e%d7%9c-%d7%9c%d7%a8%d7%9b%d7%91-thank-you-hashem-abs/",
  "sku": "madbekot-7",
  "short_description": "<p class=\"product_title entry-title elementor-heading-title elementor-size-default\"><strong>\u05de\u05d3\u05d1\u05e7\u05ea \u05e1\u05de\u05dc \u05dc\u05e8\u05db\u05d1 THANK YOU HASHEM&quot; ABS&quot;</strong></p>",
  "description": "<div id=\"model-response-message-contentr_dae2baeaeeebd6ba\" class=\"markdown markdown-main-panel\" dir=\"rtl\">\n<p><strong>\u05de\u05d3\u05d1\u05e7\u05ea \u05e4\u05e8\u05de\u05d9\u05d5\u05dd \u05d1\u05d5\u05dc\u05d8\u05ea, \u05e9\u05dc &quot;THANK YOU HASHEM&quot; &q

In [ ]:
# List the top-level fields available in each product record

print("Available product fields:\n")

for field in products[0].keys():
    print(f"- {field}")

Available product fields:

- id
- name
- slug
- parent
- type
- variation
- permalink
- sku
- short_description
- description
- on_sale
- prices
- price_html
- average_rating
- review_count
- images
- categories
- tags
- brands
- attributes
- variations
- grouped_products
- has_options
- is_purchasable
- is_in_stock
- is_on_backorder
- low_stock_remaining
- stock_availability
- sold_individually
- weight
- dimensions
- formatted_weight
- formatted_dimensions
- add_to_cart
- is_password_protected
- extensions
- _links


In [ ]:
# Inspect the pricing structure of the first product.
# The API stores price-related values inside the nested "prices" object.

print(json.dumps(products[0]["prices"], indent=2))

{
  "price": "1100",
  "regular_price": "1100",
  "sale_price": "1100",
  "price_range": null,
  "currency_code": "ILS",
  "currency_symbol": "\u20aa",
  "currency_minor_unit": 2,
  "currency_decimal_separator": ".",
  "currency_thousand_separator": ",",
  "currency_prefix": "\u20aa ",
  "currency_suffix": ""
}


In [ ]:
# Inspect how categories and brands are represented
# so we can correctly flatten them into dataset columns.

print("CATEGORIES:")
print(json.dumps(products[0]["categories"], indent=2))

print("\nBRANDS:")
print(json.dumps(products[0]["brands"], indent=2))

CATEGORIES:
[
  {
    "id": 428,
    "name": "\u05de\u05d3\u05d1\u05e7\u05d5\u05ea \u05d1\u05e8\u05e1\u05dc\u05d1",
    "slug": "%d7%9e%d7%93%d7%91%d7%a7%d7%95%d7%aa",
    "link": "https://malevegadish.com/product-category/%d7%9e%d7%93%d7%91%d7%a7%d7%95%d7%aa/"
  },
  {
    "id": 426,
    "name": "\u05de\u05d5\u05e6\u05e8\u05d9\u05dd \u05e0\u05dc\u05d5\u05d5\u05d9\u05dd",
    "slug": "%d7%9e%d7%95%d7%a6%d7%a8%d7%99%d7%9d-%d7%a0%d7%9c%d7%95%d7%95%d7%99%d7%9d",
    "link": "https://malevegadish.com/product-category/%d7%9e%d7%95%d7%a6%d7%a8%d7%99%d7%9d-%d7%a0%d7%9c%d7%95%d7%95%d7%99%d7%9d/"
  }
]

BRANDS:
[]


## 3. Product Data Extraction

The API returns product information in a nested JSON structure. Before transforming the data, we define the fields required for the final dataset and map each source field to a clean business-friendly column.

### Target Dataset Fields

| Final Column | API Source | Transformation |
|---|---|---|
| Product_ID | `id` | Direct extraction |
| Product_Name | `name` | Direct extraction |
| SKU | `sku` | Direct extraction |
| Current_Price | `prices.price` | Convert minor units to decimal price |
| Regular_Price | `prices.regular_price` | Convert minor units to decimal price |
| Currency | `prices.currency_code` | Direct extraction |
| On_Sale | `on_sale` | Direct extraction |
| Average_Rating | `average_rating` | Direct extraction |
| Review_Count | `review_count` | Direct extraction |
| Categories | `categories[].name` | Combine category names |
| Brands | `brands[].name` | Combine brand names |
| In_Stock | `is_in_stock` | Direct extraction |
| Product_URL | `permalink` | Direct extraction |

### Transformation Approach

The extraction process will:

1. Retrieve the required fields from each product.
2. Decode prices using the API's currency metadata.
3. Flatten category and brand lists into readable text.
4. Preserve missing values instead of introducing artificial values.
5. Produce a consistent tabular structure suitable for CSV and Excel export.

In [ ]:
def convert_api_price(price_value, minor_unit):
    """
    Convert an API price stored in the smallest currency unit
    into a standard decimal price.

    Example:
        "1100" with minor_unit=2 -> 11.00
    """
    if price_value in (None, ""):
        return None

    try:
        return int(price_value) / (10 ** int(minor_unit))
    except (ValueError, TypeError):
        return None


def extract_product_record(product):
    """
    Extract and flatten the fields required for the final product dataset.

    Parameters
    ----------
    product : dict
        Raw product record returned by the WooCommerce Store API.

    Returns
    -------
    dict
        Clean, flat product record suitable for a pandas DataFrame.
    """

    prices = product.get("prices", {})

    currency_minor_unit = prices.get("currency_minor_unit", 0)

    categories = product.get("categories", [])
    brands = product.get("brands", [])

    category_names = [
        category.get("name")
        for category in categories
        if category.get("name")
    ]

    brand_names = [
        brand.get("name")
        for brand in brands
        if brand.get("name")
    ]

    return {
        "Product_ID": product.get("id"),
        "Product_Name": product.get("name"),
        "SKU": product.get("sku"),

        "Current_Price": convert_api_price(
            prices.get("price"),
            currency_minor_unit
        ),

        "Regular_Price": convert_api_price(
            prices.get("regular_price"),
            currency_minor_unit
        ),

        "Currency": prices.get("currency_code"),

        "On_Sale": product.get("on_sale"),
        "Average_Rating": product.get("average_rating"),
        "Review_Count": product.get("review_count"),

        "Categories": ", ".join(category_names) if category_names else None,
        "Brands": ", ".join(brand_names) if brand_names else None,

        "In_Stock": product.get("is_in_stock"),
        "Product_URL": product.get("permalink")
    }

In [ ]:
# Transform the raw API response into flat product records.

extracted_products = [
    extract_product_record(product)
    for product in products
]

print(f"Records extracted: {len(extracted_products)}")

Records extracted: 3


In [ ]:
# Convert the extracted records into a DataFrame
# for easier inspection and downstream processing.

import pandas as pd

test_df = pd.DataFrame(extracted_products)

test_df

,Product_ID,Product_Name,SKU,Current_Price,Regular_Price,Currency,On_Sale,Average_Rating,Review_Count,Categories,Brands,In_Stock,Product_URL
0,12192,מדבקת סמל לרכב THANK YOU HASHEM&quot; ABS&quot;,madbekot-7,11.0,11.0,ILS,False,0,0,"מדבקות ברסלב, מוצרים נלווים",None,True,https://malevegadish.com/product/%d7%9e%d7%93%...
1,12184,קונטרס מעשה משבעה בעטלירס &#8211; עם ברכת המזו...,conters mease mishav'a beatalirs-1,1.5,1.5,ILS,False,0,0,"קונטרס מעשה משבעה בעטלירס, קונטרסים, קונטרסים ...",None,True,https://malevegadish.com/product/%d7%a7%d7%95%...
2,12181,קונטרס מעשה משבעה בעטלירס &#8211; עם ברכת המזו...,conters mease mishav'a beatalirs,1.5,1.5,ILS,False,0,0,"קונטרס מעשה משבעה בעטלירס, קונטרסים, קונטרסים ...",None,True,https://malevegadish.com/product/%d7%a7%d7%95%...


## 4. Pagination and Full Catalogue Extraction

The initial API request confirmed that the product endpoint is accessible and returns structured JSON data.

The full extraction process will now:

1. Request products page by page.
2. Detect the available pagination information from the API response.
3. Apply a controlled request interval to avoid excessive traffic.
4. Handle timeouts and temporary HTTP errors.
5. Preserve the original API responses for reproducibility.
6. Stop when all available product pages have been collected.
7. Record extraction statistics for validation.

In [ ]:
import requests
import time
import json
from datetime import datetime


BASE_URL = "https://malevegadish.com/wp-json/wc/store/v1/products"

# Keep the page size moderate while testing the full extraction.
RECORDS_PER_PAGE = 10

# The website requests roughly one request per second.
REQUEST_DELAY = 1.0

# Connection timeout and server response timeout.
TIMEOUT = (10, 90)


def fetch_product_page(page, per_page=RECORDS_PER_PAGE):
    """
    Retrieve one page of products from the public catalogue API.

    Returns
    -------
    tuple
        (products, response_headers)

    Raises
    ------
    requests.exceptions.RequestException
        If the request fails.
    """

    params = {
        "page": page,
        "per_page": per_page
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=TIMEOUT
    )

    response.raise_for_status()

    products = response.json()

    return products, response.headers

In [ ]:
# Test pagination metadata using the first page.

test_page, response_headers = fetch_product_page(
    page=1,
    per_page=RECORDS_PER_PAGE
)

print(f"Products returned: {len(test_page)}")

print("\nPagination headers:")
print(f"Total products: {response_headers.get('X-WP-Total')}")
print(f"Total pages: {response_headers.get('X-WP-TotalPages')}")

Products returned: 10

Pagination headers:
Total products: 324
Total pages: 33


## 5. Full Catalogue Extraction

The API reports 324 products across 33 pages using the current extraction settings.

To make the extraction reliable and reproducible, the pipeline includes:

- Page-by-page pagination
- Request pacing
- Timeout handling
- Retry logic for temporary server errors
- Progress tracking
- Extraction timestamps
- Raw JSON preservation

The original API response will be preserved before any cleaning or transformation is performed.

In [ ]:
import requests
import time
import json
from datetime import datetime


# Extraction settings
RECORDS_PER_PAGE = 10
REQUEST_DELAY = 1.0
MAX_RETRIES = 3
TIMEOUT = (10, 90)


def fetch_product_page_with_retry(page, per_page=RECORDS_PER_PAGE):
    """
    Retrieve one product page with retry handling for temporary failures.

    Parameters
    ----------
    page : int
        Page number to request.
    per_page : int
        Number of products requested per page.

    Returns
    -------
    tuple
        products, response_headers, response_time

    Raises
    ------
    requests.exceptions.RequestException
        If the request continues to fail after all retries.
    """

    params = {
        "page": page,
        "per_page": per_page
    }

    for attempt in range(1, MAX_RETRIES + 1):

        start_time = time.time()

        try:
            response = requests.get(
                BASE_URL,
                params=params,
                timeout=TIMEOUT
            )

            elapsed = time.time() - start_time

            # Retry temporary server/rate-limit responses
            if response.status_code in [429, 500, 502, 503, 504]:

                if attempt < MAX_RETRIES:
                    wait_time = attempt * 3

                    print(
                        f"Page {page}: HTTP {response.status_code}. "
                        f"Retrying in {wait_time}s..."
                    )

                    time.sleep(wait_time)
                    continue

            response.raise_for_status()

            products = response.json()

            return products, response.headers, elapsed

        except requests.exceptions.RequestException as error:

            if attempt == MAX_RETRIES:
                raise

            wait_time = attempt * 3

            print(
                f"Page {page}: Request failed. "
                f"Retrying in {wait_time}s..."
            )

            time.sleep(wait_time)

    raise requests.exceptions.RequestException(
        f"Unable to retrieve page {page}."
    )

In [ ]:
# Start the full catalogue extraction.

all_products = []
extraction_log = []

extraction_start = datetime.now()

# We already confirmed that the catalogue currently contains
# 33 pages at 10 products per page.
total_pages = 33

print(f"Starting extraction of {total_pages} pages...")
print("-" * 60)


for page in range(1, total_pages + 1):

    try:
        page_products, headers, response_time = fetch_product_page_with_retry(
            page=page
        )

        all_products.extend(page_products)

        extraction_log.append({
            "page": page,
            "records_returned": len(page_products),
            "response_time_seconds": round(response_time, 2),
            "status": "success"
        })

        print(
            f"Page {page:02d}/{total_pages} | "
            f"Records: {len(page_products):02d} | "
            f"Response: {response_time:.2f}s | "
            f"Total collected: {len(all_products)}"
        )

        # Respect the website's requested request frequency.
        if page < total_pages:
            time.sleep(REQUEST_DELAY)

    except Exception as error:

        extraction_log.append({
            "page": page,
            "records_returned": 0,
            "response_time_seconds": None,
            "status": f"failed: {str(error)}"
        })

        print(f"Page {page}: EXTRACTION FAILED")
        print(f"Error: {error}")

        break


extraction_end = datetime.now()

print("-" * 60)
print(f"Extraction completed: {extraction_end}")
print(f"Total records collected: {len(all_products)}")

Starting extraction of 33 pages...
------------------------------------------------------------
Page 01/33 | Records: 10 | Response: 6.32s | Total collected: 10
Page 02/33 | Records: 10 | Response: 6.78s | Total collected: 20
Page 03/33 | Records: 10 | Response: 7.06s | Total collected: 30
Page 04/33 | Records: 10 | Response: 4.85s | Total collected: 40
Page 05/33 | Records: 10 | Response: 4.78s | Total collected: 50
Page 06/33 | Records: 10 | Response: 4.56s | Total collected: 60
Page 07/33 | Records: 10 | Response: 4.61s | Total collected: 70
Page 08/33 | Records: 10 | Response: 4.04s | Total collected: 80
Page 09/33 | Records: 10 | Response: 5.84s | Total collected: 90
Page 10/33 | Records: 10 | Response: 6.33s | Total collected: 100
Page 11/33 | Records: 10 | Response: 3.88s | Total collected: 110
Page 12/33 | Records: 10 | Response: 5.18s | Total collected: 120
Page 13/33 | Records: 10 | Response: 6.51s | Total collected: 130
Page 14/33 | Records: 10 | Response: 5.25s | Total coll

In [ ]:
# Basic extraction validation

print(f"Expected records: 324")
print(f"Collected records: {len(all_products)}")

print("\nValidation result:")

if len(all_products) == 324:
    print("✓ Record count matches the API catalogue count.")
else:
    print("⚠ Record count does not match the expected catalogue count.")

Expected records: 324
Collected records: 324

Validation result:
✓ Record count matches the API catalogue count.


In [ ]:
# Check whether the extraction produced duplicate product IDs.

product_ids = [
    product.get("id")
    for product in all_products
]

unique_product_ids = set(product_ids)

print(f"Total extracted records: {len(product_ids)}")
print(f"Unique product IDs: {len(unique_product_ids)}")
print(f"Duplicate IDs: {len(product_ids) - len(unique_product_ids)}")

Total extracted records: 324
Unique product IDs: 324
Duplicate IDs: 0


## 6. Raw Data Preservation

Before transforming or cleaning the extracted catalogue, the original API response is saved as a raw JSON file.

This creates a reproducible workflow where the source data remains unchanged and can be revisited if any transformation or cleaning decision needs to be reviewed.

The raw dataset will not be overwritten during subsequent processing.

In [ ]:
# Save the untouched API records as a raw JSON file.
# This preserves the original structure returned by the website.

RAW_FILE = "raw_products.json"

with open(RAW_FILE, "w", encoding="utf-8") as file:
    json.dump(
        all_products,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Raw dataset saved successfully: {RAW_FILE}")
print(f"Records saved: {len(all_products)}")

Raw dataset saved successfully: raw_products.json
Records saved: 324


In [ ]:
# Verify that the saved JSON file can be read back successfully.

with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_products_check = json.load(file)

print(f"Records loaded from raw file: {len(raw_products_check)}")

if len(raw_products_check) == len(all_products):
    print("✓ Raw file validation passed.")
else:
    print("⚠ Raw file record count does not match the extracted data.")

Records loaded from raw file: 324
✓ Raw file validation passed.


In [ ]:
# Check whether the main fields required for our final dataset
# are present across the extracted products.

required_fields = [
    "id",
    "name",
    "prices",
    "categories",
    "brands",
    "permalink"
]

print("Required field availability:\n")

for field in required_fields:
    available = sum(
        1 for product in all_products
        if product.get(field) is not None
    )

    print(
        f"{field:15} : "
        f"{available}/{len(all_products)} records"
    )

Required field availability:

id              : 324/324 records
name            : 324/324 records
prices          : 324/324 records
categories      : 324/324 records
brands          : 324/324 records
permalink       : 324/324 records


## 7. Transforming Raw JSON into a Structured Dataset

The raw API response contains nested product information such as prices, categories, and brands.

To make the data suitable for analysis and delivery, the extracted records are transformed into a flat tabular structure using the field mapping defined earlier.

The original raw JSON remains unchanged. All transformations are performed on a separate dataset.

In [ ]:
# Convert each nested product record into the flat structure
# defined in the extraction function.

extracted_products = [
    extract_product_record(product)
    for product in all_products
]

df = pd.DataFrame(extracted_products)

print(f"Dataset shape: {df.shape}")
print(f"Records: {len(df)}")
print(f"Columns: {len(df.columns)}")

Dataset shape: (324, 13)
Records: 324
Columns: 13


In [ ]:
# Display the first five records to confirm that
# nested API fields were flattened correctly.

df.head()

,Product_ID,Product_Name,SKU,Current_Price,Regular_Price,Currency,On_Sale,Average_Rating,Review_Count,Categories,Brands,In_Stock,Product_URL
0,12192,מדבקת סמל לרכב THANK YOU HASHEM&quot; ABS&quot;,madbekot-7,11.0,11.0,ILS,False,0,0,"מדבקות ברסלב, מוצרים נלווים",None,True,https://malevegadish.com/product/%d7%9e%d7%93%...
1,12184,קונטרס מעשה משבעה בעטלירס &#8211; עם ברכת המזו...,conters mease mishav'a beatalirs-1,1.5,1.5,ILS,False,0,0,"קונטרס מעשה משבעה בעטלירס, קונטרסים, קונטרסים ...",None,True,https://malevegadish.com/product/%d7%a7%d7%95%...
2,12181,קונטרס מעשה משבעה בעטלירס &#8211; עם ברכת המזו...,conters mease mishav'a beatalirs,1.5,1.5,ILS,False,0,0,"קונטרס מעשה משבעה בעטלירס, קונטרסים, קונטרסים ...",None,True,https://malevegadish.com/product/%d7%a7%d7%95%...
3,12176,תיקון הכללי &quot;ישועת ישראל&quot; -סקאי לבן ...,tikon haklali &quot;yeshu'at yisral&quot; &#82...,7.5,7.5,ILS,False,0,0,"הוצאת ""כתבי הנחל"", חוברות, ישועת ישראל, ליקוטי...",None,True,https://malevegadish.com/product/%d7%aa%d7%99%...
4,12173,תיקון הכללי &quot;ישועת ישראל&quot; -סקאי חום ...,tikon haklali &quot;yeshu'at yisral&quot; &#82...,7.5,7.5,ILS,False,0,0,"הוצאת ""כתבי הנחל"", חוברות, ישועת ישראל, ליקוטי...",None,True,https://malevegadish.com/product/%d7%aa%d7%99%...


### Initial Data Structure Check

Before cleaning the dataset, we inspect the column types and non-null counts.

This helps identify fields that require type conversion, missing-value handling, or additional transformation.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324 entries, 0 to 323
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Product_ID      324 non-null    int64  
 1   Product_Name    324 non-null    object 
 2   SKU             324 non-null    object 
 3   Current_Price   324 non-null    float64
 4   Regular_Price   324 non-null    float64
 5   Currency        324 non-null    object 
 6   On_Sale         324 non-null    bool   
 7   Average_Rating  324 non-null    object 
 8   Review_Count    324 non-null    int64  
 9   Categories      323 non-null    object 
 10  Brands          0 non-null      object 
 11  In_Stock        324 non-null    bool   
 12  Product_URL     324 non-null    object 
dtypes: bool(2), float64(2), int64(2), object(7)
memory usage: 28.6+ KB


In [ ]:
# Identify missing values in each final dataset column.
# Missing values are reviewed before deciding whether
# they require cleaning or should remain as missing.

missing_values = df.isna().sum()

missing_values[missing_values > 0].sort_values(ascending=False)

,0
Brands,324
Categories,1


## 8. Data Cleaning

The extracted dataset is now structurally complete, but several fields require basic cleaning and type standardization.

The cleaning process will:

1. Convert rating values to numeric format.
2. Standardize text fields by removing unnecessary whitespace.
3. Preserve genuine missing values rather than replacing them with artificial values.
4. Review fields with high levels of missing data.
5. Prepare the dataset for validation and analysis.

No original raw data will be modified during this process.

In [ ]:
# Create a working copy so that the extracted DataFrame remains unchanged.

df_clean = df.copy()

# Convert Average_Rating from text to numeric.
# Invalid or empty values will become NaN rather than causing an error.

df_clean["Average_Rating"] = pd.to_numeric(
    df_clean["Average_Rating"],
    errors="coerce"
)

# Remove unnecessary whitespace from text fields.
text_columns = [
    "Product_Name",
    "SKU",
    "Currency",
    "Categories",
    "Product_URL"
]

for column in text_columns:
    df_clean[column] = df_clean[column].str.strip()

print("Text fields standardized.")
print("Average_Rating dtype:", df_clean["Average_Rating"].dtype)

Text fields standardized.
Average_Rating dtype: float64


In [ ]:
# Review the updated data types after cleaning.

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324 entries, 0 to 323
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Product_ID      324 non-null    int64  
 1   Product_Name    324 non-null    object 
 2   SKU             324 non-null    object 
 3   Current_Price   324 non-null    float64
 4   Regular_Price   324 non-null    float64
 5   Currency        324 non-null    object 
 6   On_Sale         324 non-null    bool   
 7   Average_Rating  324 non-null    float64
 8   Review_Count    324 non-null    int64  
 9   Categories      323 non-null    object 
 10  Brands          0 non-null      object 
 11  In_Stock        324 non-null    bool   
 12  Product_URL     324 non-null    object 
dtypes: bool(2), float64(3), int64(2), object(6)
memory usage: 28.6+ KB


In [ ]:
# Summarize missing values after the initial cleaning step.

missing_summary = (
    df_clean.isna()
    .sum()
    .to_frame("Missing_Count")
)

missing_summary["Missing_Percentage"] = (
    missing_summary["Missing_Count"]
    / len(df_clean)
    * 100
)

missing_summary = (
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Count", ascending=False)
)

missing_summary

,Missing_Count,Missing_Percentage
Brands,324,100.000000
Categories,1,0.308642


### Reviewing Missing Categories

Only one product is missing a category value.

Because the missing rate is extremely low (0.31%), the record will be inspected before deciding whether the value should remain missing or the field should be removed.

In [ ]:
# Identify the product with a missing category.

missing_category = df_clean[
    df_clean["Categories"].isna()
]

missing_category[
    [
        "Product_ID",
        "Product_Name",
        "SKU",
        "Current_Price",
        "Regular_Price",
        "Categories",
        "Product_URL"
    ]
]

,Product_ID,Product_Name,SKU,Current_Price,Regular_Price,Categories,Product_URL
7,11854,Donate to distribution,troma,148.0,148.0,None,https://malevegadish.com/product/%d7%aa%d7%a8%...


### Handling Unusable Fields

The `Brands` field contains no populated values across the extracted catalogue.

Because there is no usable brand information to analyze or deliver, the field is removed from the cleaned dataset rather than filling it with assumed values.

The original raw JSON still contains the original API structure, so this decision is fully reversible.

In [ ]:
# Remove the Brands column because it contains no usable information.

df_clean = df_clean.drop(columns=["Brands"])

print(f"Cleaned dataset shape: {df_clean.shape}")
print("Brands column removed successfully.")

Cleaned dataset shape: (324, 12)
Brands column removed successfully.


## 9. Data Quality Validation

After cleaning, the dataset is checked for structural and business-level consistency.

The first validation checks whether the extraction introduced duplicate products.

`Product_ID` is used as the primary uniqueness check because it is assigned by the source system and should identify each product uniquely.

In [ ]:
# Check for duplicate Product_ID values.

duplicate_ids = df_clean["Product_ID"].duplicated().sum()

print(f"Duplicate Product IDs: {duplicate_ids}")

if duplicate_ids == 0:
    print("✓ No duplicate products detected.")
else:
    print("⚠ Duplicate Product IDs require investigation.")

Duplicate Product IDs: 0
✓ No duplicate products detected.


In [ ]:
# Confirm the final column structure after the initial cleaning.

print("Final columns:")
for column in df_clean.columns:
    print(f"- {column}")

print(f"\nRows: {len(df_clean)}")
print(f"Columns: {len(df_clean.columns)}")

Final columns:
- Product_ID
- Product_Name
- SKU
- Current_Price
- Regular_Price
- Currency
- On_Sale
- Average_Rating
- Review_Count
- Categories
- In_Stock
- Product_URL

Rows: 324
Columns: 12


In [ ]:
# Validate key numeric and categorical fields before export.

validation_results = {
    "Negative Current Prices": (df_clean["Current_Price"] < 0).sum(),
    "Negative Regular Prices": (df_clean["Regular_Price"] < 0).sum(),
    "Negative Review Counts": (df_clean["Review_Count"] < 0).sum(),
    "Ratings Below 0": (df_clean["Average_Rating"] < 0).sum(),
    "Ratings Above 5": (df_clean["Average_Rating"] > 5).sum(),
    "Missing Product URLs": df_clean["Product_URL"].isna().sum(),
    "Missing Product Names": df_clean["Product_Name"].isna().sum(),
    "Missing SKUs": df_clean["SKU"].isna().sum(),
    "Missing Currency": df_clean["Currency"].isna().sum()
}

validation_results

{'Negative Current Prices': np.int64(0),
 'Negative Regular Prices': np.int64(0),
 'Negative Review Counts': np.int64(0),
 'Ratings Below 0': np.int64(0),
 'Ratings Above 5': np.int64(0),
 'Missing Product URLs': np.int64(0),
 'Missing Product Names': np.int64(0),
 'Missing SKUs': np.int64(0),
 'Missing Currency': np.int64(0)}

## 10. Price Consistency Validation

Before exporting the final dataset, the price fields are checked for logical consistency.

For products that are not on sale, the current price should normally match the regular price. For products marked as being on sale, the current price should not be higher than the regular price.

This check helps identify potential extraction or transformation errors before delivery.

In [ ]:
# Check whether sale status agrees with the extracted price values.

not_on_sale_mismatch = (
    (~df_clean["On_Sale"]) &
    (df_clean["Current_Price"] != df_clean["Regular_Price"])
).sum()

sale_price_mismatch = (
    df_clean["On_Sale"] &
    (df_clean["Current_Price"] > df_clean["Regular_Price"])
).sum()

print(f"Non-sale price mismatches: {not_on_sale_mismatch}")
print(f"Invalid sale prices: {sale_price_mismatch}")

if not_on_sale_mismatch == 0 and sale_price_mismatch == 0:
    print("✓ Price consistency validation passed.")
else:
    print("⚠ Price inconsistencies detected. Review affected records.")

Non-sale price mismatches: 0
Invalid sale prices: 0
✓ Price consistency validation passed.


## 11. Export Clean Dataset

The validated dataset is now exported in two standard formats:

- **CSV** — lightweight and widely compatible with data-analysis tools.
- **Excel (.xlsx)** — convenient for clients who want to review or work with the dataset in a spreadsheet.

The exported files contain the cleaned 324-record dataset and do not include the pandas index.

In [ ]:
# Export the validated dataset in both CSV and Excel formats.

CSV_FILE = "products_clean.csv"
EXCEL_FILE = "products_clean.xlsx"

df_clean.to_csv(
    CSV_FILE,
    index=False,
    encoding="utf-8-sig"
)

df_clean.to_excel(
    EXCEL_FILE,
    index=False,
    sheet_name="Products"
)

print(f"CSV exported successfully: {CSV_FILE}")
print(f"Excel exported successfully: {EXCEL_FILE}")
print(f"Rows exported: {len(df_clean)}")
print(f"Columns exported: {len(df_clean.columns)}")

CSV exported successfully: products_clean.csv
Excel exported successfully: products_clean.xlsx
Rows exported: 324
Columns exported: 12


In [ ]:
# Read the exported files back into pandas to verify that
# the saved datasets contain the expected records and columns.

csv_check = pd.read_csv(CSV_FILE)
excel_check = pd.read_excel(EXCEL_FILE)

print("CSV validation:")
print(f"Shape: {csv_check.shape}")

print("\nExcel validation:")
print(f"Shape: {excel_check.shape}")

if csv_check.shape == df_clean.shape:
    print("✓ CSV export validation passed.")

if excel_check.shape == df_clean.shape:
    print("✓ Excel export validation passed.")

CSV validation:
Shape: (324, 12)

Excel validation:
Shape: (324, 12)
✓ CSV export validation passed.
✓ Excel export validation passed.


## 12. Extraction Log

A simple extraction log is created to document the execution of the data collection process.

The log records the extraction time, number of pages processed, number of products collected, validation results, and any failed requests.

This provides a basic audit trail and makes the extraction process easier to review or reproduce.

In [ ]:
# Create a human-readable extraction log for the completed run.

LOG_FILE = "extraction_log.txt"

successful_pages = sum(
    1 for entry in extraction_log
    if entry["status"] == "success"
)

failed_pages = sum(
    1 for entry in extraction_log
    if entry["status"] != "success"
)

total_response_time = sum(
    entry["response_time_seconds"]
    for entry in extraction_log
    if entry["response_time_seconds"] is not None
)

with open(LOG_FILE, "w", encoding="utf-8") as file:

    file.write("WEB DATA EXTRACTION LOG\n")
    file.write("=" * 60 + "\n\n")

    file.write(f"Source URL: {BASE_URL}\n")
    file.write(f"Extraction started: {extraction_start}\n")
    file.write(f"Extraction completed: {extraction_end}\n\n")

    file.write("EXTRACTION SUMMARY\n")
    file.write("-" * 60 + "\n")
    file.write(f"Pages processed: {len(extraction_log)}\n")
    file.write(f"Successful pages: {successful_pages}\n")
    file.write(f"Failed pages: {failed_pages}\n")
    file.write(f"Records collected: {len(all_products)}\n")
    file.write(f"Unique Product IDs: {df_clean['Product_ID'].nunique()}\n")
    file.write(f"Total response time: {total_response_time:.2f} seconds\n\n")

    file.write("VALIDATION RESULTS\n")
    file.write("-" * 60 + "\n")
    file.write("Expected records: 324\n")
    file.write(f"Collected records: {len(all_products)}\n")
    file.write(
        "Record count validation: "
        + ("PASSED\n" if len(all_products) == 324 else "FAILED\n")
    )
    file.write(
        "Duplicate Product ID validation: "
        + ("PASSED\n" if df_clean["Product_ID"].nunique() == len(df_clean)
           else "FAILED\n")
    )
    file.write(
        "Price consistency validation: PASSED\n"
    )

    file.write("\nPAGE DETAILS\n")
    file.write("-" * 60 + "\n")

    for entry in extraction_log:
        file.write(
            f"Page {entry['page']:02d} | "
            f"Records: {entry['records_returned']:02d} | "
            f"Response: {entry['response_time_seconds']}s | "
            f"Status: {entry['status']}\n"
        )

print(f"Extraction log saved successfully: {LOG_FILE}")

Extraction log saved successfully: extraction_log.txt


In [ ]:
# Display the completed log so we can confirm that the audit information
# was written correctly.

with open(LOG_FILE, "r", encoding="utf-8") as file:
    print(file.read())

WEB DATA EXTRACTION LOG

Source URL: https://malevegadish.com/wp-json/wc/store/v1/products
Extraction started: 2026-09-14 20:30:08.847506
Extraction completed: 2026-09-14 20:33:51.015853

EXTRACTION SUMMARY
------------------------------------------------------------
Pages processed: 33
Successful pages: 33
Failed pages: 0
Records collected: 324
Unique Product IDs: 324
Total response time: 190.09 seconds

VALIDATION RESULTS
------------------------------------------------------------
Expected records: 324
Collected records: 324
Record count validation: PASSED
Duplicate Product ID validation: PASSED
Price consistency validation: PASSED

PAGE DETAILS
------------------------------------------------------------
Page 01 | Records: 10 | Response: 6.32s | Status: success
Page 02 | Records: 10 | Response: 6.78s | Status: success
Page 03 | Records: 10 | Response: 7.06s | Status: success
Page 04 | Records: 10 | Response: 4.85s | Status: success
Page 05 | Records: 10 | Response: 4.78s | Status: 

13. **Project Summary**

This project demonstrates an end-to-end web data extraction workflow using a publicly accessible WooCommerce catalogue API.

The workflow covered:

- API endpoint discovery and response validation
- Structured JSON data extraction
- Pagination handling across the full catalogue
- Retry handling for temporary request failures
- Controlled request timing
- Nested JSON flattening
- Price conversion using API currency metadata
- Text and data-type standardization
- Missing-value assessment
- Duplicate and business-rule validation
- Raw data preservation
- CSV and Excel dataset generation
- Extraction logging and audit documentation

### Final Dataset

- **Records:** 324
- **Columns:** 12
- **Unique Products:** 324
- **Duplicate Product IDs:** 0
- **Failed Extraction Pages:** 0
- **CSV Export:** Successful
- **Excel Export:** Successful

The resulting dataset is suitable for downstream analysis, reporting, catalog management, or further data-processing workflows.